# ReAct from Absolute Zero

**Prerequisite:** finish `01_tool_calling.ipynb` first.

You saw tool calling with structured JSON in notebook 01. This notebook strips **all** of that away. We use:

- The `openai` Python SDK (raw API calls)
- A prompt written as a plain Python string
- Tools defined as plain functions in a dictionary
- Regex to parse the output
- A `for` loop

That is the entire agent. No framework. No decorator. No hub. ~50 lines of Python.

In [ ]:
!uv pip install -q openai python-dotenv requests

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"
print("Key loaded.")

## 1. The prompt is just a string

This is the same ReAct format from the original ReAct paper (Yao et al., 2022), but we are not pulling it from anywhere. We are typing it ourselves. Read every line — there is nothing hidden.

In [ ]:
REACT_PROMPT = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
{scratchpad}"""

print("Prompt template length:", len(REACT_PROMPT), "characters")
print("Placeholders: {tools}, {tool_names}, {input}, {scratchpad}")

Four placeholders. That is all the "configuration" an agent needs. We fill them in with string `.format()`.

## 2. Tools are just functions in a dictionary

No decorator. No schema. No framework. A tool is a Python function, and we describe it to the model in plain English.

In [ ]:
import requests

def add(expression: str) -> str:
    """Add two integers separated by a comma."""
    a, b = [int(x.strip()) for x in expression.split(",")]
    return str(a + b)

def multiply(expression: str) -> str:
    """Multiply two integers separated by a comma."""
    a, b = [int(x.strip()) for x in expression.split(",")]
    return str(a * b)

def web_search(query: str) -> str:
    """Search the web for current information."""
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        return f"[FAKE SEARCH] no Tavily key — pretend results for: {query}"
    r = requests.post(
        "https://api.tavily.com/search",
        json={"api_key": api_key, "query": query, "max_results": 3},
        timeout=15,
    )
    r.raise_for_status()
    data = r.json()
    return "\n".join(
        f"- {item['title']}: {item['content'][:200]}"
        for item in data.get("results", [])
    )

# The registry: name -> (function, description)
TOOLS = {
    "add": (add, "Add two integers. Input: two integers separated by a comma, e.g. '17, 25'"),
    "multiply": (multiply, "Multiply two integers. Input: two integers separated by a comma, e.g. '6, 7'"),
    "web_search": (web_search, "Search the web for current information. Input: a plain search query string"),
}

# Build the text blocks the prompt needs
tools_text = "\n".join(f"{name}: {desc}" for name, (_, desc) in TOOLS.items())
tool_names = ", ".join(TOOLS.keys())

print("Tools block for the prompt:")
print(tools_text)
print()
print("Tool names:", tool_names)

No `@tool`. No JSON schema. No `bind_tools()`. The model sees a plain text list of tool names and descriptions. That is all it needs.

## 3. Format the prompt and look at what the model will receive

Let's render the prompt for one question before we call the API. This is the exact string that goes over the wire.

In [ ]:
question = "What is 12 multiplied by 7, then add 100 to that?"

first_prompt = REACT_PROMPT.format(
    tools=tools_text,
    tool_names=tool_names,
    input=question,
    scratchpad="",
)

print(first_prompt)
print()
print(f"({len(first_prompt)} characters)")

## 4. Call the OpenAI API directly

One function call. The critical parameter is `stop` — we tell the model to stop generating as soon as it writes `\nObservation:`, because the observation must come from the tool, not the model.

In [ ]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from env

def call_llm(prompt: str) -> str:
    """Send a prompt to the model and return the text. Stops at Observation."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        stop=["\nObservation:", "Observation:"],
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

output = call_llm(first_prompt)
print("Model returned:")
print(output)

That is the entire LLM integration. One function, five lines, no wrapper classes.

## 5. Parse the output with regex

The model wrote `Thought:`, `Action:`, and `Action Input:`. We extract the action and input with two regular expressions.

In [ ]:
import re

def parse_output(text: str) -> dict:
    """Parse ReAct output into either (action, action_input) or (final_answer)."""
    final = re.search(r"Final Answer:\s*(.*)", text, re.DOTALL)
    if final:
        return {"final_answer": final.group(1).strip()}

    action = re.search(r"Action:\s*(.*?)(?:\n|$)", text)
    action_input = re.search(r"Action Input:\s*(.*)", text, re.DOTALL)
    if action and action_input:
        return {
            "action": action.group(1).strip(),
            "action_input": action_input.group(1).strip(),
        }

    raise ValueError(f"Could not parse:\n{text}")

parsed = parse_output(output)
print("Parsed:", parsed)

## 6. The complete agent — 30 lines

Here is the entire ReAct agent. Read it top to bottom. There is nothing else.

In [ ]:
def react_agent(question: str, max_steps: int = 8, verbose: bool = True):
    scratchpad = ""

    for step in range(1, max_steps + 1):
        # 1. Build the prompt
        prompt = REACT_PROMPT.format(
            tools=tools_text,
            tool_names=tool_names,
            input=question,
            scratchpad=scratchpad,
        )

        # 2. Call the LLM
        text = call_llm(prompt)

        if verbose:
            print(f"\n{'='*50} Step {step} {'='*50}")
            print(text)

        # 3. Parse the response
        parsed = parse_output(text)

        # 4a. If final answer, we are done
        if "final_answer" in parsed:
            if verbose:
                print(f"\n>>> FINAL ANSWER: {parsed['final_answer']}")
            return parsed["final_answer"]

        # 4b. Otherwise, run the tool
        action = parsed["action"]
        action_input = parsed["action_input"]

        if action not in TOOLS:
            observation = f"Error: unknown tool '{action}'. Available: {tool_names}"
        else:
            fn, _ = TOOLS[action]
            try:
                observation = fn(action_input)
            except Exception as e:
                observation = f"Error: {e}"

        if verbose:
            print(f"\n  -> Ran {action}({action_input!r}) => {observation}")

        # 5. Append to scratchpad and loop
        scratchpad += f"{text}\nObservation: {observation}\n"

    print(f"\nAgent did not finish in {max_steps} steps.")
    return None

In [ ]:
react_agent("What is 12 multiplied by 7, then add 100 to that?")

In [ ]:
react_agent("Who is the CEO of OpenAI?")

## 7. Line count

Let's count how many lines of actual code this agent is (excluding blank lines and comments).

In [ ]:
import inspect

sources = [call_llm, parse_output, react_agent, add, multiply, web_search]
total = 0
for fn in sources:
    lines = [l for l in inspect.getsource(fn).split("\n") if l.strip() and not l.strip().startswith("#")]
    total += len(lines)
    print(f"  {fn.__name__}: {len(lines)} lines")
print(f"\nTotal: {total} lines of code")
print("Plus the prompt template string and the TOOLS dictionary.")
print("That is the entire agent. No framework needed.")

## Recap

You just built a complete ReAct agent from scratch with:

| Component | What it is |
|---|---|
| Prompt | A Python string with 4 placeholders |
| LLM call | `openai.chat.completions.create()` with `stop` |
| Parser | Two regex patterns |
| Tool registry | A plain dictionary |
| Agent loop | A `for` loop with an `if/else` |

There is no magic. There is no framework. There are no hidden abstractions.

Everything LangChain, LangGraph, CrewAI, or any other framework does is convenience on top of these five pieces. Now you know what is underneath.

**Next:** `03_react_failure_modes.ipynb` — we deliberately break each component of the loop and watch what happens.